# Compute large embeddings

Computes Parquet embedding datasets for the six local encoders too large to run outside Colab: bge-multilingual-gemma2, Qwen3-Embedding-8B, KaLM-Embedding-Gemma3-12B, Llama-Embed-Nemotron-8B, Harrier-OSS-v1-27B, F2LLM-v2-14B, for one source of the tree at a time: the Masoretic Psalms at the half-verse or the verse, or a scroll at the verse.

Runtime > Change runtime type > pick a GPU runtime with as much VRAM as possible.

| Model | Load dtype | Approx. memory |
| --- | --- | --- |
| bge-multilingual-gemma2 | float16 | ~18.5 GB |
| Qwen3-Embedding-8B | auto (bf16, native) | ~15.1 GB |
| KaLM-Embedding-Gemma3-12B | bfloat16 | ~23.5 GB |
| Llama-Embed-Nemotron-8B | bfloat16 | ~16 GB |
| Harrier-OSS-v1-27B | auto (native dtype) | ~54 GB (estimated, not measured; 27B params, needs >40GB VRAM) |
| F2LLM-v2-14B | bfloat16 | ~28 GB (estimated, not measured) |

A 40GB A100 fits any of bge-multilingual-gemma2, Qwen3-Embedding-8B, KaLM-Embedding-Gemma3-12B, Llama-Embed-Nemotron-8B, or F2LLM-v2-14B plus activations. A 16GB T4 fits only Qwen3-Embedding-8B or Llama-Embed-Nemotron-8B, and even those are tight. Harrier-OSS-v1-27B's estimated ~54GB exceeds a 40GB A100; it needs an 80GB A100/H100 or 8-bit loading.

Llama-Embed-Nemotron-8B is licensed for non-commercial, research use only (NVIDIA's customized-nscl-v1).


In [ ]:
!rm -rf /content/tehillim-embeddings
!git clone --depth 1 --filter=blob:none --no-checkout \
    https://github.com/rdtaylorjr/tehillim-embeddings.git /content/tehillim-embeddings
!git -C /content/tehillim-embeddings sparse-checkout set --no-cone src pyproject.toml
!git -C /content/tehillim-embeddings checkout
!cd /content/tehillim-embeddings && pip install .

Set `SCOPE_CHOICE` to the source to embed: `"bhsa-half_verse"`, `"bhsa-verse"`, or a scroll such as `"dss-11Q5-none-verse"` (the slugs `semantic.sources.SOURCES` declares). Set `MODEL_CHOICE` to `"bge"`, `"qwen3"`, `"kalm"`, `"llama-nemotron"`, `"harrier"`, or `"f2llm"` to run just one model, or leave it `None` to run all six. Set `VARIATION_CHOICE` to `"consonantal"`, `"vocalized"`, or `"cantillation"` to run just one text state, or leave it `None` to run every state the model and the source share (a scroll has consonants only). `generate` treats an already-written dataset as done, so this is safe to re-run after a partial failure.

The datasets a source reads are cloned on first use: the BHSA for the Masoretic scopes, ETCBC/dss and the tehillim-scribes Text-Fabric module for a scroll.


In [ ]:
import os
import subprocess
from pathlib import Path

from semantic.generate import generate
from semantic.large_models import ensure_corpus_data, gpu_memory_summary, models_for_choice
from semantic.sources import SOURCES

SCOPE_CHOICE = "dss-11Q5-none-verse"
MODEL_CHOICE = None
VARIATION_CHOICE = None


def _clone(url: str, destination: Path) -> None:
    subprocess.run(["git", "clone", "--depth", "1", url, str(destination)], check=True)


os.environ.update(ensure_corpus_data(data_dir=Path("/content/_corpus_data"), clone=_clone))

source = next(s for s in SOURCES if s.slug == SCOPE_CHOICE)
units = source.load()
print(f"{len(units)} units loaded from {source.scope.directory}")

output_root = Path("/content/tehillim-embeddings")
data_dir = output_root / "data"
files_before_this_run = set(data_dir.rglob("part-0.parquet")) if data_dir.exists() else set()

for slug, model_name, torch_dtype in models_for_choice(MODEL_CHOICE):
    print(f"computing {model_name} (torch_dtype={torch_dtype})...")
    written = generate(units, output_root, source, [slug], variation=VARIATION_CHOICE)
    print(f"  wrote {written}")
    summary = gpu_memory_summary()
    if summary:
        print(f"  [GPU memory] {summary}")

print("done")

Download only the Parquet files that appeared under `data/` since the cell above started (compared against the directory listing taken before it ran): the zip preserves the Hive-partitioned directory structure, so unzipping it into the repo root reproduces the correct paths directly, no manual placement needed. Works no matter how many times the cell above was re-run, since it reads real file state, not a record of which call wrote what.


In [ ]:
import zipfile

from google.colab import files

new_files = sorted(set(data_dir.rglob("part-0.parquet")) - files_before_this_run)

if not new_files:
    print("nothing new on disk since this notebook's run cell started")
else:
    with zipfile.ZipFile("/content/data.zip", "w") as zf:
        for path in new_files:
            zf.write(path, arcname=str(path.relative_to(output_root)))
    files.download("/content/data.zip")